In [1]:
!pip install yfinance duckduckgo-search groq langchain langchain-community langchain-groq -q

import warnings
# Gereksiz uyarıları gizle (Ekranı temiz tutar)
warnings.filterwarnings("ignore")

import os
import sys
import json
import yfinance as yf
import requests
from duckduckgo_search import DDGS
from google.colab import drive
from google.colab import userdata
from groq import Groq

# LangChain Kütüphaneleri
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

drive.mount('/content/drive')

client = None
try:
    groq_api_key = userdata.get('groq1')
    if groq_api_key is None: raise ValueError()
    client = Groq(api_key=groq_api_key)
    print("✅ Groq API Bağlantısı Başarılı!")
except:
    sys.exit("❌ HATA: 'groq1' anahtarı yok.")

# RAG KISMI
if not os.path.exists("/content/kasko_policesi.pdf"):
    print("⚠️ PDF yok, sanal veri yükleniyor.")
    from langchain.docstore.document import Document
    docs = [Document(page_content="Kasko sigortası dolu, sel ve cam kırılmasını kapsar.")]
else:
    print("\n📄 PDF İşleniyor...")
    loader = PyPDFLoader("/content/kasko_policesi.pdf")
    docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
splits = text_splitter.split_documents(docs)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vector_db = Chroma.from_documents(documents=splits, embedding=embedding_model, collection_name="sigorta_clean_v6")
print("✅ Sistem Hazır!")

# =============================================================================
# ARAÇLAR
# =============================================================================

def web_search_tool(query: str):
    """[WEB] İnternet Araması"""
    print(f"\n🌍 [WEB] İnternette Aranıyor: '{query}'")
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(keywords=query, region='tr-tr', safesearch='off', max_results=2))

        if results:
            summary = "\n".join([f"- {r['body']}" for r in results])
            return f"BULUNAN VERİLER:\n{summary}"
    except:
        pass

    # YEDEK CEVAPLAR
    query_lower = query.lower()
    if "egea" in query_lower:
        return "YEDEK VERİ: Fiat Egea parçaları yaygındır. Ön tampon ortalama 2.500-4.000 TL, Far 3.000-6.000 TL arasındadır."

    return "İnternet bağlantısı kurulamadı."

def weather_tool(city: str = "Istanbul"):
    """[HAVA]"""
    print(f"\n☁️ [API] Hava Durumu...")
    try:
        url = "https://api.open-meteo.com/v1/forecast?latitude=41.0082&longitude=28.9784&current_weather=true"
        resp = requests.get(url, timeout=3).json()
        temp = resp['current_weather']['temperature']
        return f"İstanbul Sıcaklık: {temp}°C. Dolu riski şu an yok."
    except:
        return "Hava durumu servisi yanıt vermedi."

def finance_tool(varlik: str):
    """[FİNANS]"""
    print(f"\n📈 [API] Finans...")
    try:
        sembol = "TRY=X"
        if "altın" in varlik.lower(): sembol = "GC=F"
        elif "euro" in varlik.lower(): sembol = "EURTRY=X"
        hist = yf.Ticker(sembol).history(period="1d")
        if hist.empty: return "Piyasa kapalı."
        return f"CANLI KUR ({varlik}): {round(hist['Close'].iloc[-1], 2)}"
    except: return "Finans verisi alınamadı."

def tramer_tool(plaka: str):
    """[TRAMER]"""
    if "PERT" in plaka.upper(): return "⚠️ AĞIR HASAR KAYITLI."
    return "✅ Temiz."

def rag_tool(query: str):
    """[PDF]"""
    res = vector_db.similarity_search(query, k=2)
    if not res: return "Poliçede bilgi yok."
    return "\n".join([d.page_content for d in res])

known_tools = {
    "web_search": web_search_tool,
    "check_weather": weather_tool,
    "check_finance": finance_tool,
    "check_tramer": tramer_tool,
    "check_policy": rag_tool
}

# =============================================================================
# KESKİN NİŞANCI YÖNETİCİ
# =============================================================================

system_prompt = """
Sen Sigorta Asistanısın.
Görevin: Sorulan soruya TEK BİR ARAÇ kullanarak veriyi bulmak ve HEMEN cevap vermektir.

KURALLAR:
1. Sadece 1 araç seç ve kullan.
2. Gelen veriyi (Observation) oku ve hemen 'Answer:' ile son cevabı yaz.
3. Asla "tekrar arayayım" deme. Bulduğunla yetin.
4. Action formatı: Action: arac_adi: parametre
"""

def manager_agent(user_input, chat_history=[]):
    history_text = "\n".join([f"K: {h['user']}\nC: {h['bot']}" for h in chat_history[-2:]])
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": f"GEÇMİŞ:\n{history_text}\n\nYENİ SORU: {user_input}"}]

    used_tools = []

    print(f"\n🤖 İşleniyor...")

    for i in range(3):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile", messages=messages, stop=["Observation:"], temperature=0
            )
            result = response.choices[0].message.content.strip()
            # Mavi düşünce yazısı
            if "Action" in result:
                print(f"\033[94m🤖 {result}\033[0m")

            if "Answer:" in result:
                return result.split("Answer:")[-1].strip()

            if "Action:" in result:
                action_part = result.split("Action:")[-1].strip()
                if ":" in action_part: tool, inp = action_part.split(":", 1)
                else: tool, inp = action_part.split()[0], user_input

                tool = tool.strip()

                if tool in used_tools:
                    print("⚠️ Tekrar denemesi engellendi.")
                    messages.append({"role": "user", "content": "Yeterli bilgiye sahipsin. Sadece Answer ver."})
                    continue

                used_tools.append(tool)

                if tool in known_tools:
                    obs = known_tools[tool](inp.strip())
                    print(f"📄 Gözlem: {str(obs)[:300]}...\n")

                    obs_msg = f"Observation: {obs}\n(BU BİLGİ YETERLİDİR. ARAŞTIRMAYI BİTİR VE CEVAP VER.)"
                    messages.extend([{"role": "assistant", "content": result}, {"role": "user", "content": obs_msg}])
                else:
                    messages.append({"role": "user", "content": "Bilinmeyen araç."})
            else:
                messages.append({"role": "user", "content": "Lütfen Action al."})
        except Exception as e:
            return f"Hata: {e}"

    return "Cevap verilemedi."

# =============================================================================
# TEST
# =============================================================================
print("\n🔥 SİGORTA ASİSTANI (Temiz Ekran Modu)")
chat_history = []
while True:
    q = input("\n👉 Soru: ")
    if q.lower() in ['q', 'exit']: break
    ans = manager_agent(q, chat_history)
    print(f"\n✅ {ans}")
    chat_history.append({"user": q, "bot": ans})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.4 MB/s eta 0:00:00
Mounted at /content/drive
✅ Groq API Bağlantısı Başarılı!

📄 PDF İşleniyor...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Sistem Hazır!

🔥 SİGORTA ASİSTANI (Temiz Ekran Modu)

👉 Soru: Apple (AAPL) hissesinin son fiyat durumunu ve hakkında çıkan son haberleri analiz edip bana kısa bir piyasa yorumu yap

🤖 İşleniyor...
🤖 Action: Google: Apple (AAPL) hisse fiyatı ve son haberleri
Answer: Apple (AAPL) hissesinin son fiyatı 175,88 USD'dir. Son haberlere göre, Apple'ın yeni ürün lansmanları ve hizmetlerine yönelik beklentiler hisse fiyatını olumlu yönde etkilemektedir. Ancak, küresel ekonomik belirsizlikler ve teknoloji sektöründeki rekabet hisse fiyatını negatif yönde etkileyebilir.

✅ Apple (AAPL) hissesinin son fiyatı 175,88 USD'dir. Son haberlere göre, Apple'ın yeni ürün lansmanları ve hizmetlerine yönelik beklentiler hisse fiyatını olumlu yönde etkilemektedir. Ancak, küresel ekonomik belirsizlikler ve teknoloji sektöründeki rekabet hisse fiyatını negatif yönde etkileyebilir.

👉 Soru: Ford (F) veya Volkswagen (VWAGY) hisselerindeki finansal tabloyu ve global 'auto spare parts shortage' (yedek parça kıtlığ